# Testes preliminares de AutoML no Google Colab

C318 — Trabalho 2 — Equipe 4. TPOT 1.1.0 e AutoGluon Tabular 1.6.3.

Este notebook reconstrói as células usadas nos testes de 12/11/2026, com os ajustes de API feitos durante a sessão. Não é a exportação original do Colab e não contém saídas inventadas. Resultados observados estão em resultados/testes_2026-11-12.md.

**Uso exploratório:** divisão 80/20 e nove classes. A validação é fornecida ao AutoGluon; a preparação do TPOT usa imputação antes da divisão. Esses procedimentos NÃO atendem ao protocolo oficial. Antes da comparação final, recuperar as partições 60/20/20 do Trabalho 1, entregar somente treino às ferramentas e ajustar transformações dentro dos folds. Este notebook não deve acessar o teste reservado do Trabalho 1.


## 1. Instalação
Execute no Colab. Se for solicitado, reinicie a sessão após instalar e continue nos imports. Os arquivos requirements registram apenas as versões das ferramentas, não todas as dependências.

In [ ]:
%pip install -q autogluon.tabular==1.6.3 tpot==1.1.0


In [ ]:
from importlib.metadata import version
import platform
import os

print("TPOT:", version("tpot"))
print("AutoGluon:", version("autogluon.tabular"))
print("Python:", platform.python_version())
print("CPUs:", os.cpu_count())


## 2. Leitura do CSV
O caminho é relativo. Se não encontrar o arquivo, o Colab solicitará upload do mesmo HPLC_data.csv do Trabalho 1.

In [ ]:
from pathlib import Path
import pandas as pd

candidatos = [Path("HPLC_data.csv"), Path("data/HPLC_data.csv"), Path("../data/HPLC_data.csv")]
csv_path = next((p for p in candidatos if p.is_file()), None)
if csv_path is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError("Coloque HPLC_data.csv em data/ ou ajuste os caminhos relativos.") from exc
    files.upload()
    csv_path = Path("HPLC_data.csv")

df = pd.read_csv(csv_path)
assert df.shape == (13031, 19), "Confira se este é o CSV original do Trabalho 1."
assert "Diagnosis" in df.columns
print("CSV carregado:", df.shape)
print(df.columns.tolist())


## 3. Preparação preliminar
Reproduz a preparação usada para verificar funcionamento. Não reproduz a limpeza nem as partições oficiais do Trabalho 1.

In [ ]:
import numpy as np
import re
from sklearn.model_selection import train_test_split

for col in ["Gender", "Religion", "Present District", "Weekness", "Jaundice", "Diagnosis"]:
    df[col] = df[col].astype("string").str.strip()

def parse_age(value):
    if pd.isna(value):
        return np.nan
    match = re.match(r"(\d+)\s*Yrs\s*(\d+)\s*month", str(value))
    if not match:
        return np.nan
    return int(match.group(1)) + int(match.group(2)) / 12

df["Age_years"] = df["Age"].apply(parse_age)
df = df.drop(columns=["Sl No", "Age"])
counts = df["Diagnosis"].value_counts()
df = df[df["Diagnosis"].isin(counts[counts >= 3].index)].copy()

train_df, valid_df = train_test_split(
    df, test_size=0.20, stratify=df["Diagnosis"], random_state=42
)
print("Treino:", train_df.shape)
print("Validação:", valid_df.shape)
print("Classes:", train_df["Diagnosis"].nunique())
print(train_df["Diagnosis"].value_counts())


## 4. AutoGluon — limite de 300 segundos
O teste original usou tuning_data. Isso deve ser removido no experimento oficial. A ferramenta pode descartar classes raras e ignorar modelos com dependências ausentes; examine o log. Cada execução usa uma nova pasta para preservar os artefatos anteriores.

In [ ]:
from autogluon.tabular import TabularPredictor
from datetime import datetime
from time import perf_counter

ag_path = Path("resultados_autogluon") / datetime.now().strftime("preliminar_%Y%m%d_%H%M%S_%f")
inicio = perf_counter()
predictor_ag = TabularPredictor(
    label="Diagnosis", problem_type="multiclass", eval_metric="f1_macro", path=str(ag_path)
).fit(train_data=train_df, tuning_data=valid_df, time_limit=300, presets="medium_quality")
tempo_ag = perf_counter() - inicio
print(f"AutoGluon finalizado em {tempo_ag / 60:.2f} minutos")
print(predictor_ag.leaderboard(valid_df, silent=True))


## 5. TPOT — preparação exploratória
A imputação abaixo reproduz a tentativa inicial e aprende com toda a base preliminar. Não usar esse procedimento para estimar desempenho oficial: imputação e codificação precisam ser ajustadas dentro dos folds de treinamento.

In [ ]:
from sklearn.impute import SimpleImputer

X = df.drop(columns=["Diagnosis"])
y = df["Diagnosis"]
X = pd.get_dummies(X, columns=["Gender", "Religion", "Present District", "Weekness", "Jaundice"], dummy_na=True)
X = X.apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
imputer = SimpleImputer(strategy="median")
X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
assert X_train.index.equals(train_df.index)
assert X_valid.index.equals(valid_df.index)
print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)


## 6. TPOT — busca de 5 minutos
API usada: verbose e scorers. Uma única tarefa paralela e limite de um minuto por avaliação foram usados após falhas anteriores. O tempo total pode exceder o limite da busca. O TPOT pode codificar os rótulos; a célula seguinte normaliza as previsões antes das métricas.

In [ ]:
from tpot import TPOTClassifier

inicio = perf_counter()
tpot = TPOTClassifier(
    max_time_mins=5, max_eval_time_mins=1, random_state=42,
    verbose=2, n_jobs=1, scorers=["f1_macro"], scorers_weights=[1]
)
tpot.fit(X_train, y_train)
tempo_tpot = perf_counter() - inicio
pred_tpot = tpot.predict(X_valid)
print(f"TPOT finalizado em {tempo_tpot / 60:.2f} minutos")


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

pred_tpot_decoded = np.asarray(pred_tpot)
if all(isinstance(v, str) for v in y_valid) and not all(isinstance(v, str) for v in pred_tpot_decoded):
    pred_tpot_decoded = tpot.label_encoder_.inverse_transform(pred_tpot_decoded.astype(int))

print(f"F1 macro: {f1_score(y_valid, pred_tpot_decoded, average='macro', zero_division=0):.4f}")
print(f"Acurácia: {accuracy_score(y_valid, pred_tpot_decoded):.4f}")
print("Pipeline encontrada:")
print(tpot.fitted_pipeline_)


## Próxima etapa

Salvar saídas reais e logs; implementar o protocolo oficial com as partições do Trabalho 1, processamento sem vazamento, classes preservadas e validação externa fora das ferramentas. As métricas deste notebook são preliminares. Consultar README e registro de resultados para os problemas identificados.
